# 02 — Forecast model comparison

Reads `reports/metrics/forecast_metrics.csv` and compares model performance across topics.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.config import METRICS_DIR

sns.set_theme(style='whitegrid')

In [ ]:
metrics = pd.read_csv(METRICS_DIR / 'forecast_metrics.csv')
metrics.head()

In [ ]:
# Average metrics across topics
summary = (
    metrics.groupby('model')[['MAE', 'RMSE', 'MAPE', 'sMAPE']]
           .mean().round(3).sort_values('MAE')
)
summary

In [ ]:
# Boxplot of MAE by model
order = summary.index.tolist()
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=metrics, x='model', y='MAE', order=order, ax=ax)
ax.set_title('MAE distribution across topics')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# How often is each model best (lowest MAE) per topic?
best_per_topic = metrics.loc[metrics.groupby('topic_id')['MAE'].idxmin()]
best_counts = best_per_topic['model'].value_counts()
best_counts.plot(kind='bar', figsize=(8, 4), color='#1f77b4')
plt.title('How often each model wins by MAE (per topic)')
plt.ylabel('# topics where best')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: model × topic, MAE
pivot = metrics.pivot(index='model', columns='topic_id', values='MAE').reindex(order)
fig, ax = plt.subplots(figsize=(min(20, 1 + 0.4 * pivot.shape[1]), 5))
sns.heatmap(pivot, annot=False, cmap='YlOrRd', ax=ax)
ax.set_title('MAE per (model, topic)')
plt.tight_layout()
plt.show()